# 03 - Regresión Lasso (Penalización $L_1$) y Selección de Variables

## Curso de Machine Learning: Modelos de Regresion

En este cuaderno estudiaremos la **Regresion Lasso** (Least Absolute Shrinkage and Selection Operator). A diferencia de Ridge, la penalizacion basada en la norma $L_1$ induce dispersión (sparsity) en el vector de parametros, anulando de manera exacta los coeficientes asociados a variables irrelevantes o redundantes. Esto convierte a Lasso en un algoritmo dual: estimador de regresion y metodo embebido de seleccion de caracteristicas.

---

### Contenidos
1. Formulacion Matematica de Lasso
2. Geometria de la Dispersion: ¿Por que la Norma $L_1$ Anula Coeficientes?
3. Algoritmo de Optimizacion: Descenso de Coordenadas y Umbralizacion Suave
4. Trazado del Camino de Regularizacion (Lasso Path)
5. Ajuste Automatico del Hiperparametro $\alpha$ mediante `LassoCV`
6. Identificacion y Descarte de Variables Irrelevantes
7. Comparativa de Coeficientes y Metricas frente a OLS y Ridge
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, lasso_path
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Formulacion Matematica de Lasso

La regresion Lasso anade a la funcion de costo de minimos cuadrados una penalizacion proporcional a la norma $L_1$ del vector de coeficientes:

$$J_{\text{Lasso}}(\beta) = \frac{1}{2n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \alpha \sum_{j=1}^p |\beta_j| = \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

Donde:
- $n$: Numero de muestras.
- $p$: Numero de variables predictoras.
- $\alpha \ge 0$: Hiperparametro que controla la intensidad de la penalizacion.
  - Para $\alpha = 0$: Coincide con OLS.
  - A medida que $\alpha$ aumenta: Se reduce la norma de los parametros y cada vez mas coeficientes se vuelven exactamente cero.
  - Para un $\alpha$ suficientemente grande: Todos los coeficientes se anulan ($\beta_j = 0$), quedando unicamente el intercepto $\beta_0 = \bar{y}$.


## 2. Geometria de la Dispersion: ¿Por que la Norma $L_1$ Anula Coeficientes?

El problema de minimizacion con penalizacion es matematicamente equivalente a un problema de optimizacion con restricciones (forma de Lagrange):

$$\min_\beta \sum_{i=1}^n (y_i - \hat{y}_i)^2 \quad \text{sujeto a} \quad \sum_{j=1}^p |\beta_j| \le t$$

- Las curvas de nivel del error cuadratico forman elipses centradas en el estimador de minimos cuadrados $\hat{\beta}_{OLS}$.
- La region de restriccion de la norma $L_1$ es un poliedro con vertices puntiagudos situados exactamente sobre los ejes de coordenadas (un rombo en 2 dimensiones, un octaedro en 3 dimensiones).
- Al expandir las elipses de error, el primer punto de contacto con la region factible ocurre casi siempre en una esquina o vertice sobre un eje coordenado, donde una o mas coordenadas valen exactamente cero ($\beta_j = 0$).
- En contraste, la region de restriccion de Ridge ($L_2$) es una esfera lisa sin esquinas, por lo que el punto de tangencia raramente cae sobre un eje coordenado.


In [ ]:
# Visualizacion grafica de la intuicion geometrica: L1 vs L2
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

b1 = np.linspace(-2, 2, 400)
b2 = np.linspace(-2, 2, 400)
B1, B2 = np.meshgrid(b1, b2)

# Centro OLS y elipses de error
ols_center = [1.2, 1.3]
Error = ((B1 - ols_center[0])**2) / 0.8 + ((B2 - ols_center[1])**2) / 0.3 + 0.5 * (B1 - ols_center[0]) * (B2 - ols_center[1])

# 1. Panel Lasso (L1)
L1_norm = np.abs(B1) + np.abs(B2)
axes[0].contour(B1, B2, L1_norm, levels=[1.0], colors=['#1f77b4'], linewidths=2.5)
axes[0].fill_between([-1, 0, 1, 0, -1], [0, 1, 0, -1, 0], color='#1f77b4', alpha=0.15)
cs_err1 = axes[0].contour(B1, B2, Error, levels=[0.2, 0.5, 0.9, 1.4, 2.1], cmap='viridis', linestyles='solid')
axes[0].plot(ols_center[0], ols_center[1], 'ro', markersize=8, label='Minimo OLS')
axes[0].plot(0, 1.0, 'go', markersize=9, label='Optimo Lasso (beta_1 = 0)')
axes[0].set_title('Lasso (Norma L1): Contacto en el Vertice (beta_1 = 0)', fontsize=11, fontweight='bold')
axes[0].set_xlabel(r'$\beta_1$', fontsize=11)
axes[0].set_ylabel(r'$\beta_2$', fontsize=11)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].legend(loc='lower left', fontsize=10)
axes[0].set_aspect('equal')

# 2. Panel Ridge (L2)
L2_norm = B1**2 + B2**2
axes[1].contour(B1, B2, L2_norm, levels=[1.0], colors=['#d62728'], linewidths=2.5)
circle = plt.Circle((0, 0), 1.0, color='#d62728', alpha=0.15)
axes[1].add_patch(circle)
cs_err2 = axes[1].contour(B1, B2, Error, levels=[0.2, 0.5, 0.9, 1.4, 2.1], cmap='viridis', linestyles='solid')
axes[1].plot(ols_center[0], ols_center[1], 'ro', markersize=8, label='Minimo OLS')
axes[1].plot(0.65, 0.76, 'mo', markersize=9, label='Optimo Ridge (beta_1 != 0, beta_2 != 0)')
axes[1].set_title('Ridge (Norma L2): Tangencia en Zona Curva (sin anularse)', fontsize=11, fontweight='bold')
axes[1].set_xlabel(r'$\beta_1$', fontsize=11)
axes[1].set_ylabel(r'$\beta_2$', fontsize=11)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].legend(loc='lower left', fontsize=10)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()


## 3. Optimizacion por Descenso de Coordenadas

Dado que el termino $|\beta_j|$ tiene una cuspide no diferenciable en $\beta_j = 0$, no existe una solucion analitica cerrada matricial como en Ridge.

Scikit-Learn optimiza Lasso mediante el algoritmo de **Descenso de Coordenadas (Coordinate Descent)**:
1. Mantiene fijos todos los coeficientes excepto uno, $\beta_j$.
2. Optimiza con respecto a $\beta_j$ utilizando el operador de **umbralizacion suave (Soft-Thresholding)**:
   $$\hat{\beta}_j = S(z_j, \alpha) = \text{sign}(z_j) \max(|z_j| - \alpha, 0)$$
   Donde $z_j$ representa la correlacion parcial residual de la variable $j$ con el error acumulado.
3. Si la importancia de la variable $|z_j| \le \alpha$, el operador fija exactamente $\beta_j = 0$.
4. Itera sucesivamente sobre cada una de las variables $j = 1, \dots, p$ hasta que todos los coeficientes convergen.


## 4. Trazado del Camino de Regularizacion (Lasso Path)

Utilizamos la funcion `lasso_path` de Scikit-Learn sobre el dataset de diabetes para observar el orden en que las variables son eliminadas conforme se incrementa $\alpha$.


In [ ]:
# Carga de datos y estandarizacion previa
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculo de la trayectoria de regularizacion
alphas_lasso, coefs_lasso, _ = lasso_path(X_train_scaled, y_train, eps=1e-3, n_alphas=150)

plt.figure(figsize=(11, 6))
for i in range(X.shape[1]):
    plt.plot(alphas_lasso, coefs_lasso[i, :], label=X.columns[i], linewidth=1.8)

plt.xscale('log')
plt.xlabel('Hiperparametro alpha (Escala Logaritmica)', fontsize=11, fontweight='bold')
plt.ylabel('Magnitud del Coeficiente', fontsize=11, fontweight='bold')
plt.title('Camino de Regularizacion de Lasso: Anulacion Progresiva de Coeficientes', fontsize=13, fontweight='bold')
plt.axhline(0, color='black', linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left", title="Variables")
plt.tight_layout()
plt.show()


## 5. Ajuste Automatico de $\alpha$ mediante `LassoCV`

Scikit-Learn ofrece `LassoCV`, el cual prueba automaticamente una cuadricula iterativa de valores de $\alpha$ mediante validacion cruzada k-fold (`cv=5`), controlando la tolerancia y el numero maximo de iteraciones para garantizar convergencia.


In [ ]:
pipeline_lasso_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso_cv', LassoCV(cv=5, random_state=42, max_iter=10000))
])

pipeline_lasso_cv.fit(X_train, y_train)

mejor_alpha_lasso = pipeline_lasso_cv.named_steps['lasso_cv'].alpha_
print(f"Mejor valor de alpha seleccionado por LassoCV: {mejor_alpha_lasso:.4f}")


## 6. Seleccion de Variables: Coeficientes Anulados vs. Activos

Inspeccionamos que variables conservan coeficientes distintos de cero y cuales han sido descartadas por completo del modelo.


In [ ]:
coefs_finales_lasso = pipeline_lasso_cv.named_steps['lasso_cv'].coef_

df_seleccion = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': coefs_finales_lasso,
    'Estado': ['Conservada' if c != 0 else 'Descartada (0)' for c in coefs_finales_lasso]
})

print("Resumen de seleccion de variables realizada por Lasso:")
print(df_seleccion.sort_values(by='Coeficiente', key=abs, ascending=False).to_string(index=False))

n_retenidas = np.sum(coefs_finales_lasso != 0)
n_descartadas = np.sum(coefs_finales_lasso == 0)
print(f"\nVariables retenidas:   {n_retenidas} de {len(X.columns)}")
print(f"Variables descartadas: {n_descartadas} de {len(X.columns)}")


## 7. Comparativa de Rendimiento: OLS vs. Ridge vs. Lasso

Evaluamos los tres modelos en el conjunto de prueba independiente para contrastar precision predictiva y parsimonia.


In [ ]:
# 1. Ajuste de OLS y Ridge con pipelines
pipeline_ols = Pipeline([('scaler', StandardScaler()), ('ols', LinearRegression())]).fit(X_train, y_train)
pipeline_ridge = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=11.5))]).fit(X_train, y_train)

# 2. Predicciones en conjunto de prueba
y_pred_ols = pipeline_ols.predict(X_test)
y_pred_ridge = pipeline_ridge.predict(X_test)
y_pred_lasso = pipeline_lasso_cv.predict(X_test)

# 3. Construccion de tabla comparativa
tabla_comparativa = pd.DataFrame({
    'Modelo': ['OLS', 'Ridge (alpha=11.5)', f'Lasso (alpha={mejor_alpha_lasso:.2f})'],
    'MSE_Test': [mean_squared_error(y_test, y_pred_ols), mean_squared_error(y_test, y_pred_ridge), mean_squared_error(y_test, y_pred_lasso)],
    'MAE_Test': [mean_absolute_error(y_test, y_pred_ols), mean_absolute_error(y_test, y_pred_ridge), mean_absolute_error(y_test, y_pred_lasso)],
    'R2_Test':  [r2_score(y_test, y_pred_ols), r2_score(y_test, y_pred_ridge), r2_score(y_test, y_pred_lasso)],
    'Variables Activas': [len(X.columns), len(X.columns), n_retenidas]
})

print("Comparativa de Desempeno en Prueba:")
print(tabla_comparativa.round(4).to_string(index=False))


In [ ]:
# Visualizacion comparativa de coeficientes: OLS vs Ridge vs Lasso
df_coef_all = pd.DataFrame({
    'Variable': X.columns,
    'OLS': pipeline_ols.named_steps['ols'].coef_,
    'Ridge': pipeline_ridge.named_steps['ridge'].coef_,
    'Lasso': coefs_finales_lasso
})

x = np.arange(len(X.columns))
width = 0.25

plt.figure(figsize=(13, 5.5))
plt.bar(x - width, df_coef_all['OLS'], width=width, label='OLS', color='#d95f02')
plt.bar(x, df_coef_all['Ridge'], width=width, label='Ridge', color='#2b5c8f')
plt.bar(x + width, df_coef_all['Lasso'], width=width, label='Lasso', color='#1b9e77')

plt.xticks(x, X.columns, fontsize=11)
plt.ylabel('Valor del Coeficiente Estandarizado', fontsize=11, fontweight='bold')
plt.title('Comparacion de Coeficientes Estandarizados: OLS vs. Ridge vs. Lasso', fontsize=13, fontweight='bold')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 8. Conclusiones

### Puntos Clave:
1. **Seleccion Embebida**: Lasso no solo reduce la varianza de los coeficientes, sino que anula variables irrelevantes de forma automatica gracias a la geometria de las esquinas de la norma $L_1$.
2. **Optimizacion Numerica**: Requiere Descenso de Coordenadas con operadores de umbralizacion suave debido a la no diferenciabilidad en cero.
3. **Limitacion de Lasso con Multicolinealidad Estricta**: Cuando dos o mas variables estan fuertemente correlacionadas entre si (como las variables sanguineas `s1` y `s2`), Lasso tiende a seleccionar arbitrariamente una de ellas y descartar el resto, ignorando posibles interacciones sinergicas.

En el siguiente cuaderno, **04_ElasticNet_y_Comparativa_Modelos.ipynb**, combinaremos lo mejor de ambos mundos: la seleccion de caracteristicas de Lasso y la estabilidad frente a colinealidades de Ridge mediante la formulacion de **ElasticNet**.
